# Evolução diária da umidade: NASA POWER × SONDA-MD

Sobrepõe as médias diárias das duas bases tratadas. Os dias com cobertura parcial da SONDA-MD são destacados e não recebem interpolação.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def localizar_dados_tratados():
    for base in (Path.cwd(), *Path.cwd().parents):
        candidato = base / '2026' / 'dados_tratados'
        if ((candidato / 'sonda_md_horario_padronizado.csv').exists()
                and (candidato / 'nasa_power_horario_padronizado.csv').exists()):
            return candidato.resolve()
    raise FileNotFoundError('Não foi encontrada a pasta 2026/dados_tratados.')

PASTA_DADOS = localizar_dados_tratados()
PASTA_SAIDA = PASTA_DADOS.parent / 'resultados' / 'figuras' / 'comparacao_bases'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

def ler_umidade(nome_arquivo):
    dados = pd.read_csv(PASTA_DADOS / nome_arquivo, parse_dates=['data_hora'])
    dados['umidade_relativa'] = pd.to_numeric(dados['umidade_relativa'], errors='coerce')
    return (dados[['data_hora', 'umidade_relativa']]
            .dropna(subset=['data_hora'])
            .sort_values('data_hora')
            .drop_duplicates('data_hora'))

sonda = ler_umidade('sonda_md_horario_padronizado.csv')
nasa = ler_umidade('nasa_power_horario_padronizado.csv')
diaria_sonda = (sonda.set_index('data_hora')['umidade_relativa']
                .resample('D')
                .agg(media_umidade_sonda_md='mean', observacoes_sonda_md='count'))
diaria_nasa = (nasa.set_index('data_hora')['umidade_relativa']
               .resample('D')
               .agg(media_umidade_nasa_power='mean', observacoes_nasa_power='count'))
diaria = diaria_nasa.join(diaria_sonda, how='outer')
dias_parciais = diaria.index[diaria['observacoes_sonda_md'] < 24]
print(f'Dias representados: {len(diaria):,}')
print(f'Dias da SONDA-MD com cobertura parcial: {len(dias_parciais):,}')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6), dpi=150)
ax.plot(diaria.index, diaria['media_umidade_nasa_power'], color='#2166ac', linewidth=1.2, label='NASA POWER')
ax.plot(diaria.index, diaria['media_umidade_sonda_md'], color='#b2182b', linewidth=1.2, label='SONDA-MD')
for dia in dias_parciais:
    ax.axvspan(dia, dia + pd.Timedelta(days=1), color='#e6a700', alpha=0.18, linewidth=0)
ax.set_title('Evolução diária da umidade relativa: NASA POWER e SONDA-MD')
ax.set_xlabel('Data')
ax.set_ylabel('Umidade relativa (%)')
ax.legend(title='Fonte')
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()

arquivo_figura = PASTA_SAIDA / 'comparacao_media_diaria_umidade_nasa_power_sonda_md.png'
arquivo_tabela = PASTA_SAIDA / 'medias_diarias_umidade_nasa_power_sonda_md.csv'
fig.savefig(arquivo_figura, dpi=300, bbox_inches='tight')
diaria.to_csv(arquivo_tabela, date_format='%Y-%m-%d')
print(f'Figura salva em: {arquivo_figura}')
print(f'Tabela salva em: {arquivo_tabela}')
plt.show()